# 01 – Tổng quan dữ liệu

Mục tiêu notebook:
- Nạp dữ liệu gốc, kiểm tra kích thước, kiểu dữ liệu.
- Kiểm tra giá trị thiếu, trùng lặp, tính liên tục của chuỗi thời gian.
- Xem nhanh biến mục tiêu `Appliances`.

In [ ]:
import sys
from pathlib import Path

# Cho phép import src/ khi chạy notebook từ thư mục notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import FIGURES_DIR, TARGET, ensure_dirs
from src.data_loader import basic_report, load_raw_data

ensure_dirs()
pd.set_option("display.max_columns", 40)

## 1. Nạp dữ liệu

In [ ]:
df = load_raw_data()
print("Kích thước:", df.shape)
df.head()

In [ ]:
df.info()

## 2. Kiểm tra chất lượng dữ liệu

In [ ]:
report = basic_report(df)
pd.Series(report, name="giá trị").to_frame()

**Nhận xét:**
- Không có giá trị thiếu và không có dòng trùng lặp.
- Các mốc thời gian cách đều 10 phút, không bị đứt quãng → có thể tạo đặc trưng trễ (lag) an toàn ở các bước sau.

## 3. Thống kê mô tả

In [ ]:
df.describe().T.round(2)

## 4. Kiểm tra hai biến ngẫu nhiên rv1, rv2

In [ ]:
print("rv1 trùng hoàn toàn rv2:", (df["rv1"] == df["rv2"]).all())
print("Tương quan rv1 với Appliances:", round(df["rv1"].corr(df[TARGET]), 4))

`rv1` và `rv2` giống hệt nhau và gần như không tương quan với biến mục tiêu. Nhóm sẽ giữ **một** biến (`rv1`)
làm "biến đối chứng" khi phân tích độ quan trọng đặc trưng, và loại `rv2` vì trùng lặp.

## 5. Biến mục tiêu `Appliances` – nhìn nhanh

In [ ]:
print(df[TARGET].describe())
print("Độ lệch (skewness):", round(df[TARGET].skew(), 2))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(df["date"], df[TARGET], linewidth=0.4)
ax.set_title("Điện năng tiêu thụ của thiết bị theo thời gian")
ax.set_xlabel("Thời gian")
ax.set_ylabel("Appliances (Wh)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_appliances_timeseries.png", dpi=150)
plt.show()

**Nhận xét ban đầu:**
- Biến mục tiêu lệch phải mạnh: trung vị 60 Wh nhưng có đỉnh tới 1080 Wh.
- Chuỗi có dạng "nền thấp + các đỉnh nhọn" – các đỉnh nhiều khả năng do hành vi sử dụng thiết bị.
- Phân tích chi tiết phân phối và xu hướng theo giờ/ngày sẽ làm trong notebook EDA (`02_eda.ipynb`).